# Problem Statement: Building a Custom BPE Tokenizer Using WikiText-2 Dataset

You are tasked with building a custom Byte Pair Encoding (BPE) tokenizer from scratch using
the WikiText-2 dataset, available on Hugging Face at Salesforce/wikitext. The tokenizer should
be optimized for modeling English text and usable in downstream language modeling tasks.

## Imports

In [1]:
import re
from collections import Counter
from datasets import load_dataset, Dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast, AutoTokenizer

c:\Users\raad\Desktop\GenAI-Pinnacle-Program\17_Training_LLMs_from_Scratch\Assignments\01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Solution

1. Load and Explore the Dataset:
    - Use the wikitext dataset from Hugging Face's datasets library.
    - Select the WikiText-2 subset ("wikitext-2-v1"). This will contain 44.8k rows.

In [2]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

In [3]:
# Combine a sample of the text to scan
text_sample = " ".join(dataset["train"]["text"][:5000])

# Find all tokens like <unk>, <eos>, etc.
special_tokens = re.findall(r'<[a-zA-Z/]+>', text_sample)
token_counts = Counter(special_tokens)

print("Found Special Tokens:", token_counts)

Found Special Tokens: Counter({'<unk>': 7411})


2. Data Cleaning & Preprocessing:
    - Perform deduplication of the training text to eliminate exact duplicates.
    - Normalize and clean the text if necessary (e.g., remove special tokens like <unk> or unnecessary whitespace).

In [4]:
def clean_text(examples):
    # Process a batch of text
    cleaned_texts = []
    for text in examples["text"]:
        # Remove <unk> tokens
        text = text.replace("<unk>", "")
        
        # Remove extra whitespace, tabs, and newlines
        # .split() without arguments splits by any whitespace and removes empty strings
        text = " ".join(text.split())
        
        cleaned_texts.append(text)
    
    return {"text": cleaned_texts}

dataset = dataset.map(clean_text, batched=True)

dataset = dataset.filter(lambda x: len(x["text"]) > 0)

In [5]:
train_df = dataset["train"].to_pandas()

print(f"Original size: {len(train_df)}")

train_df = train_df.drop_duplicates(subset=['text'], keep='first')

print(f"Final size: {len(train_df)}")

dataset['train'] = Dataset.from_pandas(train_df)

del train_df

Original size: 23732
Final size: 21329


3. Tokenizer Training:
    - Implement or use Hugging Face's tokenizers library to build a BPE tokenizer from
scratch.
    - Define and train the tokenizer on the deduplicated training set.
    - Specify parameters like:
        - Vocabulary size (e.g., 30,000 tokens)
        - Special tokens: [PAD], [UNK], [CLS], [SEP], [MASK]

In [6]:
# 1. Initialize an empty BPE model
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

# 2. Set the pre-tokenizer to split text on whitespace
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel() # Critical for 100% consistency

# 3. Configure the trainer with desired vocab size and special tokens
trainer = trainers.BpeTrainer(
    vocab_size=30000, 
    min_frequency=2, 
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

# 4. Create an iterator to feed the dataset in batches
def batch_iterator(batch_size=1000):
    for i in range(0, len(dataset["train"]), batch_size):
        yield dataset["train"][i : i + batch_size]["text"]

# 5. Train the tokenizer from the iterator
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

4. Tokenizer Evaluation:
    - Test the trained tokenizer on the validation and test splits of WikiText-2.
    - Report metrics such as:
        - Vocabulary size
        - Tokenization consistency
        - Average tokens per sentence
        - Compression ratio

In [24]:
def check_consistency(dataset_split, tokenizer):
    inconsistencies = 0
    total = 0
    
    for example in dataset_split["text"]:
        if not example.strip():
            continue
        
        # 1. Encode text to IDs
        ids = tokenizer.encode(example).ids
        # 2. Decode IDs back to text
        decoded = tokenizer.decode(ids)
        
        # Note: BPE often normalizes whitespace, so we strip both for comparison
        if example.strip() != decoded.strip():
            inconsistencies += 1
        total += 1
        
    success_rate = ((total - inconsistencies) / total)
    return success_rate


def evaluate_tokenizer(split_name, dataset_split, tokenizer):
    # Filter empty lines
    valid_texts = [t for t in dataset_split["text"] if t.strip()]
    
    # Tokenize the entire split
    encoded_batches = tokenizer.encode_batch(valid_texts)
    
    total_tokens = sum(len(encoding.ids) for encoding in encoded_batches)
    total_chars = sum(len(text) for text in valid_texts)
    num_sentences = len(valid_texts)
    
    # Metrics calculation
    consistency_score = check_consistency(dataset_split, tokenizer)
    avg_tokens = total_tokens / num_sentences
    compression_ratio = total_chars / total_tokens  # Characters per token
    
    return {
        "Total Vocabulary Size": tokenizer.get_vocab_size(),
        "Split": split_name,
        "Tokenization Consistency": f"{consistency_score:.1%}",
        "Avg Tokens/Sentence": round(avg_tokens, 2),
        "Compression Ratio": round(compression_ratio, 2)
    }


In [25]:
# Run evaluation
val_metrics = evaluate_tokenizer("Validation", dataset["validation"], tokenizer)

print("\n".join([f"{k:<25}: {v}" for k, v in val_metrics.items()]))

Total Vocabulary Size    : 30000
Split                    : Validation
Tokenization Consistency : 100.0%
Avg Tokens/Sentence      : 85.98
Compression Ratio        : 4.91


5. Save and Reuse:
    - Save the tokenizer in Hugging Face-compatible format.
    - Demonstrate reloading and using the tokenizer for encoding/decoding text.

In [9]:
# 1. Wrap the trained tokenizer
# Note: Ensure you define the special tokens used during training
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    mask_token="[MASK]"
)

# 2. Save to a directory
save_path = "./wikitext-bpe-tokenizer"
hf_tokenizer.save_pretrained(save_path)

print(f"Tokenizer saved to {save_path}")


Tokenizer saved to ./wikitext-bpe-tokenizer


In [27]:
# 1. Load the tokenizer from the local directory
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

# 2. Demonstrate Encoding
text = dataset["test"][15]["text"]
encoded = reloaded_tokenizer(text)
print(f"Encoded IDs: {encoded['input_ids']}")

# 3. Demonstrate Decoding
decoded_text = reloaded_tokenizer.decode(encoded['input_ids'])
print(f"Decoded Text: {decoded_text}")

# Verify consistency
print(f"Matches Original: {text == decoded_text}")


Encoded IDs: [16934, 14649, 292, 306, 521, 435, 316, 4842, 449, 10533, 14649, 420, 2827, 449, 420, 435, 6114, 20, 290, 243, 184, 4704, 2827, 2984, 204, 186, 14194, 7572, 196, 8189, 277, 4342, 292, 4342, 7475, 290, 189, 310, 285, 5195, 949, 186, 3608, 204, 186, 2827, 10236, 196, 1382, 3608, 15740, 243, 215, 5003, 322, 1732, 262, 184, 1913, 2576, 21498, 189, 424, 310, 4425, 3449, 215, 1398, 186, 5098, 7204, 529, 196, 1382, 1166, 189, 1048, 186, 3436, 1732, 189, 243, 16352, 288, 186, 655, 28351, 204, 189, 212, 322, 1197, 929, 788, 342, 184, 564, 204, 2249, 8875, 10574, 196]
Decoded Text: Du Fu ( Wade – Giles : Tu Fu ; Chinese : ; – 770 ) was a prominent Chinese poet of the Tang dynasty . Along with Li ( Li Po ) , he is frequently called the greatest of the Chinese poets . His greatest ambition was to serve his country as a successful civil servant , but he proved unable to make the necessary accommodations . His life , like the whole country , was devastated by the An Rebellion of , and h

In [11]:
def verify_consistency(dataset_split, tokenizer):
    inconsistencies = 0
    total = 0
    
    # Iterate through the test set
    for example in dataset_split:
        original_text = example["text"]
        
        # Skip empty lines to focus on actual data
        if not original_text.strip():
            continue
            
        # 1. Encode
        ids = tokenizer.encode(original_text)
        
        # 2. Decode
        decoded_text = tokenizer.decode(ids)
        
        # 3. Compare (Note: ByteLevel BPE should match exactly)
        if original_text != decoded_text:
            inconsistencies += 1
            # Optional: Print the first error for debugging
            if inconsistencies == 1:
                print(f"First mismatch found!\nOriginal: {repr(original_text)}\nDecoded:  {repr(decoded_text)}")
        
        total += 1

    success_rate = ((total - inconsistencies) / total) * 100
    return success_rate


In [12]:
# Run on the test set
test_score = verify_consistency(dataset["test"], reloaded_tokenizer)
print(f"Final Test Consistency Score: {test_score:.2f}%")

Final Test Consistency Score: 100.00%
